# SCBE Zero-Cost 0.5B Adapter Lane

Free Colab fallback for `scbe-zero-cost-local-0.5b`. It clones the active branch, trains a compact LoRA adapter from the consolidated SCBE buckets, and packages the adapter for download. Hugging Face upload is optional and disabled by default.

In [ ]:
import os, pathlib, subprocess, sys, json

REPO_URL = "https://github.com/issdandavis/SCBE-AETHERMOORE.git"
BRANCH = "chore/repo-launch-restructure"
REPO_DIR = pathlib.Path("/content/SCBE-AETHERMOORE")

if REPO_DIR.exists():
    subprocess.check_call(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH, "--depth", "1"])
    subprocess.check_call(["git", "-C", str(REPO_DIR), "checkout", "FETCH_HEAD"])
else:
    subprocess.check_call(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)])

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
subprocess.check_call(["git", "log", "-1", "--oneline"])
print("repo", REPO_DIR)

In [ ]:
!python -m pip install -q "transformers>=4.46" "peft>=0.13" "accelerate>=1.0" "datasets>=3.0" "bitsandbytes>=0.44" "safetensors"

In [ ]:
import json, pathlib, subprocess, torch

PROFILE_ID = "scbe-zero-cost-local-0.5b"
PROFILE_PATH = pathlib.Path("config/model_training/scbe-zero-cost-local-0.5b.json")
profile = json.loads(PROFILE_PATH.read_text(encoding="utf-8"))

print("profile", profile["profile_id"])
print("base_model", profile["base_model"])
print("cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0))
    print("vram_gb", round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 2))

plan = subprocess.check_output([
    sys.executable,
    "scripts/scbe-system-cli.py",
    "model",
    "preflight",
    "--profile",
    PROFILE_ID,
    "--json",
], text=True)
print(plan)

In [ ]:
from datasets import concatenate_datasets, load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, DataCollatorForLanguageModeling, Trainer, TrainingArguments
import json, pathlib, torch

train_files = [str(pathlib.Path(p)) for p in profile["dataset"]["train_files"]]
eval_files = [str(pathlib.Path(p)) for p in profile["dataset"]["eval_files"]]
missing = [p for p in train_files + eval_files if not pathlib.Path(p).exists()]
if missing:
    raise FileNotFoundError("missing dataset files: " + ", ".join(missing[:5]))

train_ds = concatenate_datasets([load_dataset("json", data_files=p, split="train") for p in train_files])
eval_ds = concatenate_datasets([load_dataset("json", data_files=p, split="train") for p in eval_files])
print("rows", {"train": len(train_ds), "eval": len(eval_ds)})

def row_to_text(row):
    if row.get("messages"):
        return "\n".join(f"{m.get('role', 'user')}: {m.get('content', '')}" for m in row["messages"])
    if row.get("prompt") and row.get("completion"):
        return f"system: {profile['system_prompt']}\nuser: {row['prompt']}\nassistant: {row['completion']}"
    if row.get("text"):
        return str(row["text"])
    return json.dumps(row, ensure_ascii=False)

tokenizer = AutoTokenizer.from_pretrained(profile["base_model"], use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

max_length = int(profile["training"].get("max_seq_length", 1024))
def add_text(row):
    return {"text": row_to_text(row)}

def tokenize_batch(batch):
    return tokenizer(batch["text"], max_length=max_length, truncation=True)

train_text = train_ds.map(add_text)
eval_text = eval_ds.map(add_text)
tokenized_train = train_text.map(tokenize_batch, batched=True, remove_columns=list(train_text.column_names))
tokenized_eval = eval_text.map(tokenize_batch, batched=True, remove_columns=list(eval_text.column_names))

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(profile["base_model"], quantization_config=bnb_config, device_map="auto")
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=int(profile["training"].get("lora_rank", 16)),
    lora_alpha=int(profile["training"].get("lora_alpha", 32)),
    lora_dropout=float(profile["training"].get("lora_dropout", 0.05)),
    target_modules=profile["training"].get("target_modules"),
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

out_dir = pathlib.Path("/content/scbe-zero-cost-local-0.5b-lora")
args = TrainingArguments(
    output_dir=str(out_dir),
    max_steps=int(profile["training"].get("max_steps", 250)),
    per_device_train_batch_size=int(profile["training"].get("batch_size", 1)),
    gradient_accumulation_steps=int(profile["training"].get("gradient_accumulation_steps", 8)),
    learning_rate=float(profile["training"].get("learning_rate", 2e-4)),
    warmup_ratio=float(profile["training"].get("warmup_ratio", 0.03)),
    logging_steps=int(profile["training"].get("logging_steps", 10)),
    save_steps=int(profile["training"].get("save_steps", 50)),
    save_total_limit=int(profile["training"].get("save_total_limit", 2)),
    fp16=torch.cuda.is_available(),
    report_to="none",
    optim="paged_adamw_8bit",
)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)
result = trainer.train()
metrics = trainer.evaluate()
metrics.update(result.metrics)
trainer.save_model(str(out_dir))
tokenizer.save_pretrained(str(out_dir))
(out_dir / "scbe_training_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print(json.dumps(metrics, indent=2))
print("adapter", out_dir)

In [ ]:
import pathlib, shutil

out_dir = pathlib.Path("/content/scbe-zero-cost-local-0.5b-lora")
zip_path = shutil.make_archive("/content/scbe-zero-cost-local-0.5b-lora", "zip", out_dir)
print("adapter_dir", out_dir)
print("zip", zip_path)
print("Download the zip from Colab Files, or mount Drive before this cell and copy it there.")

In [ ]:
# Optional upload. Leave disabled unless you intentionally want this adapter on Hugging Face.
# import os, subprocess
# os.environ["HF_TOKEN"] = "paste-token-here"
# subprocess.check_call(["huggingface-cli", "upload", "issdandavis/scbe-zero-cost-local-0p5b-lora", "/content/scbe-zero-cost-local-0.5b-lora", ".", "--repo-type", "model"])
